# 🧪 Brain Tumor Classification - Model Testing

Test your trained models on sample images and compare results across different architectures.

## Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image as keras_image

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.classification.inference import TumorClassifier
from src.utils.constants import AVAILABLE_MODELS, CLASS_NAMES

print(f"✅ Project Root: {project_root}")

## 📁 Discover Available Trained Models

In [ ]:
import glob
from datetime import datetime

def find_all_weights():
    """Find all trained model weights in the weights directory."""
    weights_dir = project_root / "weights" / "classification"
    
    # Find all .keras files
    weight_files = list(weights_dir.glob("**/*.keras"))
    
    models_info = []
    for weight_file in weight_files:
        # Get file info
        file_size = weight_file.stat().st_size / (1024**2)  # MB
        mod_time = datetime.fromtimestamp(weight_file.stat().st_mtime)
        
        # Extract model name
        for model_name in AVAILABLE_MODELS:
            if model_name in weight_file.name:
                models_info.append({
                    'name': model_name,
                    'path': weight_file,
                    'size_mb': file_size,
                    'modified': mod_time,
                    'is_timestamped': len(weight_file.parts) > len(weights_dir.parts) + 1
                })
                break
    
    return sorted(models_info, key=lambda x: x['modified'], reverse=True)

# Find all weights
available_weights = find_all_weights()

print("╔" + "="*80 + "╗")
print("║" + " "*25 + "🤖 AVAILABLE TRAINED MODELS" + " "*27 + "║")
print("╠" + "="*80 + "╣")

if not available_weights:
    print("║  ⚠️  No trained models found! Train a model first." + " "*29 + "║")
else:
    for i, model in enumerate(available_weights, 1):
        run_type = "[Timestamped Run]" if model['is_timestamped'] else "[Latest]"
        print(f"║ {i}. {model['name']:<20} {run_type:<18} {model['size_mb']:>6.1f} MB ║")
        print(f"║    Modified: {model['modified'].strftime('%Y-%m-%d %H:%M:%S')}" + " "*42 + "║")

print("╚" + "="*80 + "╝")
print(f"\n📊 Total models found: {len(available_weights)}")

## 🎯 Select Model to Test

In [ ]:
# Select model by index (change this number)
selected_index = 1  # ← Change this to select different model (0 = first, 1 = second, etc.)

if not available_weights:
    print("❌ No models available! Please train a model first.")
else:
    selected_model = available_weights[selected_index]
    
    print("\n" + "="*60)
    print(f"✅ Selected Model: {selected_model['name']}")
    print(f"📁 Path: {selected_model['path']}")
    print(f"📅 Last Modified: {selected_model['modified'].strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"💾 Size: {selected_model['size_mb']:.1f} MB")
    print("="*60)

## 🔧 Load Model

In [ ]:
# Load the classifier
classifier = TumorClassifier(
    str(selected_model['path']),
    model_name=selected_model['name']
)

print(f"✅ {selected_model['name']} loaded successfully!")

## 🖼️ Find Test Images

In [ ]:
# Find sample test images
test_dir = project_root / "data" / "brisc2025" / "classification_task" / "test"

def get_sample_images(num_per_class=2):
    """Get sample images from each class."""
    samples = {}
    
    for class_name in CLASS_NAMES:
        class_dir = test_dir / class_name
        if class_dir.exists():
            images = list(class_dir.glob("*.jpg"))[:num_per_class]
            samples[class_name] = images
    
    return samples

sample_images = get_sample_images(num_per_class=3)

print("📸 Sample images found:")
for class_name, images in sample_images.items():
    print(f"   {class_name}: {len(images)} images")

## 🧪 Test on Single Image

In [ ]:
# Select an image to test
test_class = 'pituitary'  # ← Change this: 'glioma', 'meningioma', 'no_tumor', 'pituitary'
test_image_idx = 0      # ← Change this to test different images (0, 1, 2, ...)

test_image_path = sample_images[test_class][test_image_idx]

# Predict
result = classifier.predict(str(test_image_path))

# Load and display image
img = keras_image.load_img(test_image_path, target_size=(224, 224))

plt.figure(figsize=(10, 6))

# Display image
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.axis('off')
correct = "✅" if result['class'] == test_class else "❌"
plt.title(f"{correct} Predicted: {result['class']}\nConfidence: {result['confidence']:.1f}%", 
          fontsize=12, fontweight='bold')

# Display probabilities
plt.subplot(1, 2, 2)
probs = result['probabilities']
classes = list(probs.keys())
values = list(probs.values())

colors = ['green' if c == result['class'] else 'lightblue' for c in classes]
bars = plt.barh(classes, values, color=colors)
plt.xlabel('Confidence (%)', fontsize=10)
plt.title('Class Probabilities', fontsize=12, fontweight='bold')
plt.xlim(0, 100)

# Add percentage labels
for bar, val in zip(bars, values):
    plt.text(val + 2, bar.get_y() + bar.get_height()/2, 
             f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Print results
print(f"\n{'='*60}")
print(f"True Label:      {test_class}")
print(f"Predicted:       {result['class']}")
print(f"Confidence:      {result['confidence']:.2f}%")
print(f"Correct:         {result['class'] == test_class}")
print(f"{'='*60}")
print(result['warning'])

## 📊 Test on Multiple Images (Grid View)

In [ ]:
def test_multiple_images(num_per_class=2):
    """Test model on multiple images and display results in a grid."""
    
    all_images = []
    for class_name, images in sample_images.items():
        for img_path in images[:num_per_class]:
            all_images.append((class_name, img_path))
    
    # Create grid
    n_images = len(all_images)
    n_cols = 4
    n_rows = (n_images + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    correct_count = 0
    
    for idx, (true_class, img_path) in enumerate(all_images):
        # Predict
        result = classifier.predict(str(img_path))
        
        # Load image
        img = keras_image.load_img(img_path, target_size=(224, 224))
        
        # Display
        axes[idx].imshow(img)
        axes[idx].axis('off')
        
        # Title with result
        is_correct = result['class'] == true_class
        correct_count += is_correct
        
        color = 'green' if is_correct else 'red'
        symbol = "✅" if is_correct else "❌"
        
        axes[idx].set_title(
            f"{symbol} True: {true_class}\nPred: {result['class']} ({result['confidence']:.0f}%)",
            fontsize=10, color=color, fontweight='bold'
        )
    
    # Hide unused subplots
    for idx in range(len(all_images), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.suptitle(f"Model: {selected_model['name']} | Accuracy: {correct_count}/{n_images} ({100*correct_count/n_images:.1f}%)",
                 fontsize=14, fontweight='bold', y=1.002)
    plt.show()
    
    return correct_count, n_images

# Test on 2 images per class
correct, total = test_multiple_images(num_per_class=2)
print(f"\n📊 Test Results: {correct}/{total} correct ({100*correct/total:.1f}% accuracy)")

## 🔄 Compare Multiple Models

In [ ]:
def compare_models_on_image(image_path, models_to_compare=None):
    """Compare predictions from multiple models on the same image."""
    
    if models_to_compare is None:
        # Use all available latest models (non-timestamped)
        models_to_compare = [m for m in available_weights if not m['is_timestamped']][:5]
    
    if not models_to_compare:
        print("No models to compare!")
        return
    
    # Load image once
    img = keras_image.load_img(image_path, target_size=(224, 224))
    
    # Create subplots
    n_models = len(models_to_compare)
    fig, axes = plt.subplots(1, n_models + 1, figsize=(4*(n_models+1), 4))
    
    # Show original image
    axes[0].imshow(img)
    axes[0].axis('off')
    axes[0].set_title('Input Image', fontsize=12, fontweight='bold')
    
    # Test each model
    results = []
    for idx, model_info in enumerate(models_to_compare):
        # Load classifier
        clf = TumorClassifier(str(model_info['path']), model_name=model_info['name'])
        
        # Predict
        result = clf.predict(str(image_path))
        results.append(result)
        
        # Plot probabilities
        probs = result['probabilities']
        classes = list(probs.keys())
        values = list(probs.values())
        
        colors = ['green' if c == result['class'] else 'lightblue' for c in classes]
        axes[idx + 1].barh(classes, values, color=colors)
        axes[idx + 1].set_xlim(0, 100)
        axes[idx + 1].set_title(
            f"{model_info['name']}\n{result['class']} ({result['confidence']:.0f}%)",
            fontsize=10, fontweight='bold'
        )
        axes[idx + 1].set_xlabel('Confidence (%)', fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    return results

# Compare models on a test image
if available_weights:
    test_img = sample_images['no_tumor'][2]  # Change class/index as needed
    results = compare_models_on_image(test_img)
else:
    print("No models available for comparison!")

## 📁 Test on Your Own Image

In [ ]:
# Specify your own image path here
custom_image_path = "path/to/your/image.jpg"  # ← Change this!

# Uncomment and run after setting the path
# if Path(custom_image_path).exists():
#     result = classifier.predict(custom_image_path)
#     
#     img = keras_image.load_img(custom_image_path, target_size=(224, 224))
#     plt.figure(figsize=(8, 6))
#     plt.imshow(img)
#     plt.axis('off')
#     plt.title(f"Predicted: {result['class']} ({result['confidence']:.1f}%)", 
#               fontsize=14, fontweight='bold')
#     plt.show()
#     
#     print(f"\n{result['warning']}")
# else:
#     print(f"Image not found: {custom_image_path}")

## 📝 Summary

**What you can do with this notebook:**
1. ✅ View all available trained models
2. ✅ Select any model to test
3. ✅ Test on individual images with detailed predictions
4. ✅ Test on multiple images in a grid view
5. ✅ Compare predictions from different models
6. ✅ Test on your own custom images

**Model Location:**
- Weights: `weights/classification/`
- Training logs: `logs/classification/`